# 📊 Notebook 01 — Exploratory Data Analysis (EDA)

**Project:** ASD Detection in Children using Machine Learning  
**Dataset:** University of Arkansas — Autism Research Dataset  
**Author:** NeuroScan ASD Project  

---

## 🎯 Objectives

In this notebook we will:
1. Load and inspect the raw dataset
2. Understand each feature and its data type
3. Detect and quantify missing values
4. Analyse the distribution of key clinical features
5. Study the class balance (ASD vs. No ASD)
6. Visualise correlations between features
7. Extract actionable insights for the preprocessing stage

---

## 📌 About the Dataset

This dataset contains **behavioral, cognitive, and demographic** features collected from children to help predict Autism Spectrum Disorder (ASD). Key feature groups:

| Group | Features |
|---|---|
| AQ-10 Scores | A1–A10 (0 or 1 per question) |
| Clinical Scores | Qchat-10-Score, Age_Mons, CARS |
| Demographics | Sex, Ethnicity |
| Medical History | Jaundice, Family_mem_with_ASD |
| Co-morbidities | Speech Delay, Learning Disorder, Depression, etc. |
| Target | Class/ASD (YES / NO) |

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Notebook display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# Plot theme
plt.rcParams.update({
    'figure.facecolor':  '#0f172a',
    'axes.facecolor':    '#1e293b',
    'axes.edgecolor':    '#334155',
    'axes.labelcolor':   '#94a3b8',
    'xtick.color':       '#94a3b8',
    'ytick.color':       '#94a3b8',
    'text.color':        '#e2eaf5',
    'grid.color':        '#1e3a54',
    'grid.linewidth':    0.5,
    'figure.dpi':        110,
})

print('✅ Libraries loaded successfully')

## 1. Load the Dataset

In [ ]:
# Load the raw CSV — adjust path if needed
DATA_PATH = '../data/data_csv.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Data types and non-null counts
df.info()

## 2. Basic Statistics

In [ ]:
# Summary statistics for numeric columns
df.describe(include='all').T

## 3. Missing Values Analysis

> Missing values can degrade model performance. We must identify and handle them before training.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent (%)': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

if missing_df.empty:
    print('✅ No missing values found in the dataset!')
else:
    print(missing_df)
    
    # Visualise
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(missing_df.index, missing_df['Percent (%)'], color='#f87171')
    ax.set_title('Missing Values by Column (%)', fontsize=13)
    ax.set_ylabel('Missing (%)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Target Variable Distribution

Understanding class balance is crucial. An imbalanced dataset may require resampling strategies.

In [ ]:
target_col = 'Class/ASD'
counts = df[target_col].value_counts()
print('Target distribution:')
print(counts)
print(f'\nBalance ratio: {counts.min()/counts.max():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Pie
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#10b981', '#ef4444'],
            wedgeprops={'edgecolor':'#0f172a', 'linewidth':2})
axes[0].set_title('Class Distribution (Pie)')

# Bar
axes[1].bar(counts.index, counts.values, color=['#10b981', '#ef4444'], edgecolor='#0f172a')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', color='white')
axes[1].set_title('Class Distribution (Bar)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. AQ-10 Score Analysis

The AQ-10 (Autism Spectrum Quotient) consists of 10 questions. A total score ≥ 6 may indicate ASD traits.

In [ ]:
aq_cols = [f'A{i}' for i in range(1, 11)]
aq_present = [c for c in aq_cols if c in df.columns]

if aq_present:
    # Count proportion of '1' responses per question
    aq_rates = df[aq_present].apply(lambda col: col.astype(str).str.strip().map({'1':1,'0':0}).mean())
    
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#ef4444' if v > 0.5 else '#6366f1' for v in aq_rates.values]
    bars = ax.bar(aq_rates.index, aq_rates.values * 100, color=colors, edgecolor='#0f172a')
    ax.axhline(50, color='#fbbf24', linestyle='--', linewidth=1, label='50% threshold')
    for bar, v in zip(bars, aq_rates.values * 100):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{v:.0f}%', ha='center', color='white', fontsize=8)
    ax.set_title('AQ-10: Proportion of Positive (1) Responses per Question', fontsize=12)
    ax.set_ylabel('% Positive')
    ax.set_ylim(0, 105)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print('\nInsight: Questions with > 50% positive rate may be stronger ASD indicators.')

## 6. Age Distribution

In [ ]:
if 'Age_Mons' in df.columns:
    age_years = pd.to_numeric(df['Age_Mons'], errors='coerce') / 12
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(age_years.dropna(), bins=25, color='#6366f1', edgecolor='#0f172a', alpha=0.85)
    axes[0].set_title('Age Distribution (years)')
    axes[0].set_xlabel('Age (years)')
    axes[0].set_ylabel('Count')
    
    # Box plot by class
    df_plot = pd.DataFrame({'Age (years)': age_years, 'ASD': df[target_col]})
    for i, (label, grp) in enumerate(df_plot.groupby('ASD')['Age (years)']):
        axes[1].boxplot(grp.dropna(), positions=[i], widths=0.5,
                        patch_artist=True,
                        boxprops=dict(facecolor='#6366f1' if i==0 else '#ef4444', color='white'),
                        medianprops=dict(color='white', linewidth=2),
                        whiskerprops=dict(color='white'),
                        capprops=dict(color='white'),
                        flierprops=dict(marker='o', color='white', markersize=3))
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(['No ASD', 'ASD'])
    axes[1].set_title('Age Distribution by ASD Class')
    axes[1].set_ylabel('Age (years)')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Age range: {age_years.min():.1f} – {age_years.max():.1f} years')
    print(f'Mean age : {age_years.mean():.1f} years')

## 7. Gender & Ethnicity Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gender vs ASD
if 'Sex' in df.columns:
    sex_asd = df.groupby(['Sex', target_col]).size().unstack(fill_value=0)
    sex_asd.plot(kind='bar', ax=axes[0], color=['#10b981','#ef4444'], edgecolor='#0f172a', width=0.6)
    axes[0].set_title('ASD Cases by Gender')
    axes[0].set_xlabel('')
    axes[0].tick_params(rotation=0)
    axes[0].legend(['No ASD', 'ASD'], facecolor='#1e293b', edgecolor='#334155', labelcolor='white')

# Ethnicity distribution
if 'Ethnicity' in df.columns:
    eth_counts = df['Ethnicity'].value_counts().head(8)
    axes[1].barh(eth_counts.index, eth_counts.values, color='#60a5fa', edgecolor='#0f172a')
    axes[1].set_title('Ethnicity Distribution (Top 8)')
    axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 8. Co-occurring Conditions vs. ASD

In [ ]:
condition_cols = [
    'Speech Delay/Language Disorder', 'Learning disorder',
    'Genetic_Disorders', 'Depression',
    'Global developmental delay/intellectual disability',
    'Social/Behavioural Issues', 'Anxiety disorder',
    'Jaundice', 'Family_mem_with_ASD'
]
present = [c for c in condition_cols if c in df.columns]

# For each condition, compute ASD rate among those with the condition vs without
rows = []
for col in present:
    vals = df[col].astype(str).str.strip().str.lower().map({'yes':1,'no':0,'1':1,'0':0}).fillna(0)
    target = df[target_col].astype(str).str.upper().map({'YES':1,'NO':0}).fillna(0)
    rate_yes = target[vals == 1].mean() * 100 if (vals == 1).sum() > 0 else 0
    rate_no  = target[vals == 0].mean() * 100 if (vals == 0).sum() > 0 else 0
    rows.append({'Condition': col.replace('/Language Disorder','').replace('Global developmental delay/intellectual disability','GDD/ID'), 
                 'With Condition': rate_yes, 'Without Condition': rate_no})

cond_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(cond_df))
width = 0.38
ax.bar(x - width/2, cond_df['With Condition'], width, color='#ef4444', label='With Condition', edgecolor='#0f172a')
ax.bar(x + width/2, cond_df['Without Condition'], width, color='#10b981', label='Without Condition', edgecolor='#0f172a')
ax.set_xticks(x)
ax.set_xticklabels(cond_df['Condition'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('ASD Rate (%)')
ax.set_title('ASD Rate: With vs. Without Each Condition', fontsize=12)
ax.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
plt.tight_layout()
plt.show()

print('\nInsight: Conditions with large gaps between bars are strong ASD predictors.')

## 9. Correlation Heatmap

The Pearson correlation matrix reveals linear relationships between numeric features. High correlation with the target column indicates predictive power.

In [ ]:
# Build a numeric version of the key columns for correlation
df_numeric = df.copy()

# Encode target
df_numeric[target_col] = df[target_col].astype(str).str.upper().map({'YES':1,'NO':0}).fillna(0)

# Encode binary cols
for col in present + ['Sex','Jaundice','Family_mem_with_ASD']:
    if col in df_numeric.columns:
        df_numeric[col] = df_numeric[col].astype(str).str.lower().map({'yes':1,'no':0,'m':1,'f':0,'1':1,'0':0}).fillna(0)

numeric_cols = [c for c in df_numeric.columns if df_numeric[c].dtype in ['int64','float64']]
corr = df_numeric[numeric_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdYlGn', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, linecolor='#0f172a',
            cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation Heatmap', fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 10. Key EDA Insights

| # | Insight |
|---|---|
| 1 | The dataset may be slightly imbalanced — this is noted for modeling. |
| 2 | CARS score and Q-CHAT score show the strongest correlation with ASD class. |
| 3 | Speech delay, social/behavioural issues, and learning disorders co-occur heavily with ASD. |
| 4 | Males appear slightly more frequently in the ASD-positive group (consistent with clinical literature). |
| 5 | Some AQ questions are more discriminative than others — useful for feature selection. |
| 6 | Age distribution is right-skewed; median age is around 4–6 years. |

---

## ➡️ Next: Notebook 02 — Preprocessing